# My Code 

In [ ]:
Main controlled baseline:
50% majority + square-root BCE + no augmentation + Dynamic 1:1 sampler + seed 42

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from timm.models.layers.helpers import to_2tuple
import timm
import random
import io
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt
import os
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report
from sklearn.metrics import precision_score, recall_score, f1_score

# =========================
# Reproducibility control for controlled ablation
# =========================

SEED = 42

def set_seed(seed=42):
    import os
    import random
    import numpy as np
    import torch

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # More deterministic CUDA behaviour
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

print(f"Controlled experiment seed fixed at: {SEED}")

# =========================
# Reproduction Config
# =========================

RUN_NAME = "controlled_hard_seed42_50_sqr_no_aug_f1select"
SEED = 42

# DICC project paths
PROJECT_ROOT = Path("/home/user/jiangjie/Jiangjie_Project")
RP50_DIR = PROJECT_ROOT / "data" / "ResearchProject_50"
CHECKPOINT_PATH = PROJECT_ROOT / "weights" / "ctranspath.pth"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "kee_reproduction" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Kee baseline settings
NUM_CLASSES = 12
EPOCHS = 50
LEARNING_RATE = 1e-5
BATCH_SIZE = 128
PATIENCE = 3
NUM_WORKERS = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")
print(f"RUN_NAME: {RUN_NAME}")
print(f"RP50_DIR exists: {RP50_DIR.exists()} -> {RP50_DIR}")
print(f"CHECKPOINT_PATH exists: {CHECKPOINT_PATH.exists()} -> {CHECKPOINT_PATH}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

# 12 OED morphological feature labels used in Kee CTransPath baseline
label_columns = [
    "Irregular epithelial stratification",
    "Loss of polarity of basal cells",
    "Drop shaped rete ridges",
    "Premature keratinization in single cells",
    "Loss of epithelial cell cohesion",
    "Abnormal variation in nuclear size",
    "Abnormal variation in nuclear shape",
    "Abnormal variation in cell size",
    "Abnormal variation in cell shape",
    "Increased N:C ratio",
    "Increased number and size of nucleoli",
    "Hyperchromasia",
]

print("Number of labels:", len(label_columns))

class PathBlur(object):
    def __init__(self, blur_sigma=(0.5, 2.0), poisson_scale=(5, 20), jpeg_quality=(30, 90)):
        self.blur_sigma = blur_sigma
        self.poisson_scale = poisson_scale
        self.jpeg_quality = jpeg_quality

    def __call__(self, img):
        # 1️⃣ Gaussian Blur
        sigma = random.uniform(*self.blur_sigma)
        img = img.filter(ImageFilter.GaussianBlur(radius=sigma))

        # 2️⃣ Poisson Noise
        np_img = np.array(img).astype(np.float32)
        scale = random.uniform(*self.poisson_scale)
        noisy = np.random.poisson(np_img * scale) / scale
        noisy = np.clip(noisy, 0, 255).astype(np.uint8)
        img = Image.fromarray(noisy)

        # 3️⃣ JPEG Compression Artifact
        buffer = io.BytesIO()
        quality = random.randint(*self.jpeg_quality)
        img.save(buffer, format='JPEG', quality=quality)
        img = Image.open(buffer)

        return img

# Fixed pre-processing applied to all data (Resize, ToTensor, Normalize)
base_transforms = [
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
]

# Validation/Test Transform (No Augmentation)
val_test_transform = transforms.Compose(base_transforms)

# Define different augmentation combinations
AUGMENTATION_COMBINATIONS = {
    'None': val_test_transform, # Baseline: No Augmentation

    # 'Random_Rotation': transforms.Compose([
    #     transforms.RandomApply(
    #         [transforms.RandomRotation(degrees=(-45, 45))],
    #         p=0.5    # 50% chance to apply rotation
    #     ),
    #     *base_transforms
    # ]),
    # # 'Random_Translation': transforms.Compose([
    # #     transforms.RandomAffine(
    # #         degrees=0,                # no rotation
    # #         translate=(70/224, 70/224)  # assuming images are 224×224
    # #     ),
    # #     *base_transforms
    # # ]),

    # 'Random_Cropping': transforms.Compose([
    #     transforms.RandomApply([
    #         transforms.RandomResizedCrop(
    #             size=224,
    #             scale=(0.7, 1.0),
    #             ratio=(1.0, 1.0)
    #         )
    #     ], p=0.5),  # 50% chance to randomly crop
    #     *base_transforms
    # ]),

    # 'Random_Flipping': transforms.Compose([
    #     transforms.RandomHorizontalFlip(p=0.5),  # 50% chance to flip horizontally
    #     transforms.RandomVerticalFlip(p=0.5),    # 50% chance to flip vertically
    #     *base_transforms
    # ]),

    # 'Color_Augmentation': transforms.Compose([
    #     transforms.RandomApply([
    #         transforms.ColorJitter(
    #             brightness=0.1,
    #             contrast=0.1,
    #             saturation=0.1,
    #             hue=0.02
    #         ),
    #     ], p=0.5),
    #     *base_transforms
    # ]),

    # # 'Stain_Augmentation': transforms.Compose([
    # #     transforms.ColorJitter(
    # #         brightness=0.8,
    # #         contrast=0.8,
    # #         saturation=0.8,
    # #         hue=0.2
    # #     ),
    # #     transforms.RandomGrayscale(p=0.2),
    # #     *base_transforms
    # # ]),

    # # 'PathBlur_Augmentation': transforms.Compose([
    # #     PathBlur(
    # #         blur_sigma=(0.5, 2.0),
    # #         poisson_scale=(5, 20),
    # #         jpeg_quality=(30, 90)
    # #     ),
    # #     *base_transforms
    # # ]),

    # # 'Random_Cropping + Color_Augmentation': transforms.Compose([
    # #     transforms.RandomApply([
    # #         transforms.RandomResizedCrop(
    # #             size=224,
    # #             scale=(0.7, 1.0),
    # #             ratio=(1.0, 1.0)
    # #         ),
    # #         transforms.ColorJitter(
    # #             brightness=0.1,
    # #             contrast=0.1,
    # #             saturation=0.1,
    # #             hue=0.02
    # #         ),
    # #     ], p=0.5),
    # #     *base_transforms
    # # ]),

    # 'Random_Flipping + Color_Augmentation': transforms.Compose([
    #     transforms.RandomApply([
    #         transforms.RandomHorizontalFlip(p=0.5),
    #         transforms.RandomVerticalFlip(p=0.5),
    #         transforms.ColorJitter(
    #             brightness=0.1,
    #             contrast=0.1,
    #             saturation=0.1,
    #             hue=0.02
    #         ),
    #     ], p=0.5),
    #     *base_transforms
    # ])
}

/home/user/jiangjie/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Controlled experiment seed fixed at: 42
Using device: cuda
RUN_NAME: controlled_hard_seed42_50_sqr_no_aug_f1select
RP50_DIR exists: True -> /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50
CHECKPOINT_PATH exists: True -> /home/user/jiangjie/Jiangjie_Project/weights/ctranspath.pth
OUTPUT_DIR: /home/user/jiangjie/Jiangjie_Project/outputs/kee_reproduction/controlled_hard_seed42_50_sqr_no_aug_f1select
Number of labels: 12


In [2]:
# =========================
# Transform setting: Run A = no augmentation
# =========================

base_transforms = [
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
]

# Validation/Test Transform: no augmentation
val_test_transform = transforms.Compose(base_transforms)

# Run A: 50% Square-root Weight, no augmentation
AUGMENTATION_COMBINATIONS = {
    "None": val_test_transform
}

print("Augmentation setting:")
print(AUGMENTATION_COMBINATIONS)

Augmentation setting:
{'None': Compose(
    Resize(size=224, interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
)}


In [3]:
# @title Original code

# --- 1. CTransPath Architecture (Swin + ConvStem) ---

class ConvStem(nn.Module):

    def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=768, norm_layer=None, flatten=True):
        super().__init__()

        assert patch_size == 4
        assert embed_dim % 8 == 0

        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = (img_size[0] // patch_size[0], img_size[1] // patch_size[1])
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.flatten = flatten


        stem = []
        input_dim, output_dim = 3, embed_dim // 8
        for l in range(2):
            stem.append(nn.Conv2d(input_dim, output_dim, kernel_size=3, stride=2, padding=1, bias=False))
            stem.append(nn.BatchNorm2d(output_dim))
            stem.append(nn.ReLU(inplace=True))
            input_dim = output_dim
            output_dim *= 2
        stem.append(nn.Conv2d(input_dim, embed_dim, kernel_size=1))
        self.proj = nn.Sequential(*stem)

        self.norm = norm_layer(embed_dim) if norm_layer else nn.Identity()

    def forward(self, x):
        B, C, H, W = x.shape
        assert H == self.img_size[0] and W == self.img_size[1], \
            f"Input image size ({H}*{W}) doesn't match model ({self.img_size[0]}*{self.img_size[1]})."
        x = self.proj(x)
        if self.flatten:
            x = x.flatten(2).transpose(1, 2)  # BCHW -> BNC
        x = self.norm(x)
        return x

def ctranspath(num_classes, checkpoint_path=CHECKPOINT_PATH):
    # Load the Swin-Tiny model structure and inject the ConvStem
    model = timm.create_model(
        model_name="swin_tiny_patch4_window7_224",
        embed_layer=ConvStem, # Your custom class defined elsewhere
        pretrained=False,       # Instructs timm to download and load the weights
        num_classes=0, # Remove default head
    )

    # --- MANUAL CHECKPOINT LOADING ---
    if checkpoint_path:
        print(f"Loading weights manually from: {checkpoint_path}")
        # Assuming the checkpoint is a dict with the model state under the key 'model'
        state_dict = torch.load(checkpoint_path, map_location='cpu')

        # We need to filter the state_dict because the head layer dimensions won't match
        # CTransPath checkpoints usually save the entire model
        # We try to load the full state dict and ignore the mismatched head
        model.load_state_dict(state_dict, strict=False)

    # Ensure Global Average Pooling is explicitly enabled for feature extraction
    model.global_pool = 'avg'
    in_features = model.num_features

    # Replace the classification head for your 10 classes
    model.head = nn.Linear(in_features, num_classes)

    return model

    # # --- MANUAL CHECKPOINT LOADING ---
    # if checkpoint_path:
    #     print(f"Loading weights manually from: {checkpoint_path}")
    #     # Assuming the checkpoint is a dict with the model state under the key 'model'
    #     state_dict = torch.load(checkpoint_path, map_location='cpu')

    #     # We need to filter the state_dict because the head layer dimensions won't match
    #     # CTransPath checkpoints usually save the entire model
    #     model.load_state_dict(state_dict, strict=False)

    # # Replace the classification head for your 10 classes
    # # This is a standard fine-tuning step.
    # in_features = model.head.in_features
    # model.head = nn.Linear(in_features, num_classes)
    # return model

# def ctranspath(num_classes, checkpoint_path=CHECKPOINT_PATH):
#     # 1. Create the model structure
#     model = timm.create_model(
#         model_name="swin_tiny_patch4_window7_224",
#         embed_layer=ConvStem,
#         pretrained=False,
#         num_classes=0, # Remove default head
#     )

#     # 2. Load Checkpoint (Backbone Weights)
#     if checkpoint_path:
#         print(f"Loading weights manually from: {checkpoint_path}")
#         state_dict = torch.load(checkpoint_path, map_location='cpu')
#         # Filter state_dict to match non-strict loading if needed
#         model.load_state_dict(state_dict['model'], strict=False)

#     # 3. FREEZE THE BACKBONE (Crucial for Small Data)
#     print("❄️ Freezing ALL CTransPath backbone layers...")
#     for param in model.parameters():
#         param.requires_grad = False

#     # # 4. UNFREEZE THE LAST BLOCK (The "Fine-Tuning" Step)
#     # # This allows the model to learn specific histology textures
#     # print("🔓 Unfreezing the last Swin Transformer block...")

#     # # Unfreeze the last layer block (layers.3 in Swin Tiny)
#     # for param in model.layers[-1].parameters():
#     #     param.requires_grad = True

#     # # Unfreeze the final normalization layer
#     # for param in model.norm.parameters():
#     #     param.requires_grad = True

#     # 5. Add the Head (Trainable)
#     model.global_pool = 'avg'
#     in_features = model.num_features

#     model.head = nn.Sequential(
#         nn.Linear(in_features, 256),
#         nn.ReLU(),
#         nn.Dropout(p=0.5),
#         nn.Linear(256, num_classes)
#     )
#     # Ensure head is trainable
#     for param in model.head.parameters():
#         param.requires_grad = True

#     return model

# --- Helper Functions ---
def to_2tuple(x):
    if isinstance(x, (tuple, list)):
        return tuple(x)
    return (x, x)

def collate_fn(batch):
    # Filters out samples where image loading failed (returned None)
    batch = [item for item in batch if item[0] is not None]
    if not batch: return None, None
    return torch.utils.data.dataloader.default_collate(batch)


In [4]:
# =========================
# Model output shape check
# =========================

model = ctranspath(NUM_CLASSES, checkpoint_path=CHECKPOINT_PATH).to(device)
model.eval()

dummy = torch.randn(2, 3, 224, 224).to(device)

with torch.no_grad():
    out = model(dummy)

print("Model output shape:", out.shape)
assert out.shape == (2, NUM_CLASSES), f"Unexpected output shape: {out.shape}"

print("Model output shape check passed.")

/home/user/jiangjie/.local/lib/python3.10/site-packages/torch/functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3637.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Loading weights manually from: /home/user/jiangjie/Jiangjie_Project/weights/ctranspath.pth
Model output shape: torch.Size([2, 12])
Model output shape check passed.


In [5]:
# --- 3. Custom Dataset ---

class MultiLabelTileDataset(Dataset):
    def __init__(self, df, label_columns, transform=None):
        self.df = df.reset_index(drop=True)
        self.label_columns = label_columns
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Image path used directly from the DataFrame (assuming full path)
        img_path = self.df.loc[idx, 'filepath']

        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            print(f"File not found: {img_path}")
            return None, None

        labels = self.df.loc[idx, self.label_columns].values.astype('float32')
        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(labels, dtype=torch.float32)

        return image, labels

In [6]:
# =========================
# Load 50% hard-label split: Kee CTransPath setting
# =========================

base_path = str(RP50_DIR)
weight_type = "Sqr"
covertype = "50"

# Train files used by Kee CTransPath baseline
train_files = [Path(base_path) / f"final_df_train{i}.csv" for i in range(1, 11)]
val_file = Path(base_path) / "final_df_val.csv"
test_file = Path(base_path) / "final_df_test.csv"

print("Checking CSV files:")
for f in train_files + [val_file, test_file]:
    print(f.name, "exists:", f.exists())

# Load CSVs
train_dfs = []
for f in train_files:
    df = pd.read_csv(f, low_memory=False)
    df["source_file"] = f.name
    train_dfs.append(df)

train_df = pd.concat(train_dfs, ignore_index=True)
val_df = pd.read_csv(val_file, low_memory=False)
test_df = pd.read_csv(test_file, low_memory=False)

print("\nLoaded data:")
print("train_df:", train_df.shape)
print("val_df:", val_df.shape)
print("test_df:", test_df.shape)

# Make sure label columns are numeric 0/1
for df in [train_df, val_df, test_df]:
    for col in label_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

# Path replacement if CSV contains Colab-style path
old_prefix = "/content/ResearchProject_50"
new_prefix = str(RP50_DIR)

for df in [train_df, val_df, test_df]:
    df["filepath"] = df["filepath"].astype(str).str.replace(old_prefix, new_prefix, regex=False)

print("\nExample filepath after replacement:")
print(train_df["filepath"].iloc[0])
print("Exists:", Path(train_df["filepath"].iloc[0]).exists())

# WSI-level split check
train_slides = set(train_df["slide_name"].dropna().unique())
val_slides = set(val_df["slide_name"].dropna().unique())
test_slides = set(test_df["slide_name"].dropna().unique())

print("\nUnique WSIs:")
print("train:", len(train_slides))
print("val:", len(val_slides))
print("test:", len(test_slides))
print("all:", len(train_slides | val_slides | test_slides))

print("\nOverlap check:")
print("train ∩ val :", sorted(train_slides & val_slides))
print("train ∩ test:", sorted(train_slides & test_slides))
print("val ∩ test  :", sorted(val_slides & test_slides))

assert len(train_slides & val_slides) == 0
assert len(train_slides & test_slides) == 0
assert len(val_slides & test_slides) == 0
assert len(label_columns) == NUM_CLASSES

print("\nSplit check passed: WSI-level split with no overlap.")

Checking CSV files:
final_df_train1.csv exists: True
final_df_train2.csv exists: True
final_df_train3.csv exists: True
final_df_train4.csv exists: True
final_df_train5.csv exists: True
final_df_train6.csv exists: True
final_df_train7.csv exists: True
final_df_train8.csv exists: True
final_df_train9.csv exists: True
final_df_train10.csv exists: True
final_df_val.csv exists: True
final_df_test.csv exists: True

Loaded data:
train_df: (1359460, 22)
val_df: (198083, 21)
test_df: (314284, 21)

Example filepath after replacement:
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/266-DP-14/266-DP-14_tile_0000000.jpg
Exists: True

Unique WSIs:
train: 19
val: 6
test: 6
all: 31

Overlap check:
train ∩ val : []
train ∩ test: []
val ∩ test  : []

Split check passed: WSI-level split with no overlap.


In [7]:
# =========================
# Dynamic 1:1 sampler + square-root pos_weight check
# =========================

# Define abnormal patch: at least one of the 12 labels is positive
train_df["is_abnormal_12"] = (train_df[label_columns].sum(axis=1) > 0).astype(int)

n_abnormal = int(train_df["is_abnormal_12"].sum())
n_normal = int(len(train_df) - n_abnormal)

print("Training patch distribution:")
print("Total train patches:", len(train_df))
print("Normal train patches:", n_normal)
print("Abnormal train patches:", n_abnormal)
print("Abnormal percentage:", round(n_abnormal / len(train_df) * 100, 4), "%")

# Dynamic 1:1 sampling weights
sample_weights = np.zeros(len(train_df), dtype=np.float32)

sample_weights[train_df["is_abnormal_12"].values == 1] = 1.0 / n_abnormal
sample_weights[train_df["is_abnormal_12"].values == 0] = 1.0 / n_normal

# Each epoch contains all abnormal patches and an equal number of sampled normal patches
num_train_samples = int(n_abnormal * 2)

sampler_generator = torch.Generator()
sampler_generator.manual_seed(SEED)

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=num_train_samples,
    replacement=True,
    generator=sampler_generator
)

print("\nDynamic 1:1 sampler:")
print("num_train_samples per epoch:", num_train_samples)
print("Expected abnormal samples per epoch:", n_abnormal)
print("Expected normal samples per epoch:", n_abnormal)

# Square-root pos_weight calculation based on effective balanced epoch
positive_counts = train_df[label_columns].sum(axis=0).astype(float)

# Effective negatives under balanced epoch setting
effective_total = num_train_samples
effective_negatives = effective_total - positive_counts

raw_pos_weight = effective_negatives / (positive_counts + 1e-6)
sqrt_pos_weight = np.sqrt(raw_pos_weight)

pos_weight_tensor = torch.tensor(
    sqrt_pos_weight.values,
    dtype=torch.float32
).to(device)

weight_table = pd.DataFrame({
    "OED_feature": label_columns,
    "positive_count_train": positive_counts.values.astype(int),
    "raw_pos_weight": raw_pos_weight.values,
    "sqrt_pos_weight": sqrt_pos_weight.values
})

print("\nSquare-root pos_weight table:")
display(weight_table)

print("\npos_weight tensor shape:", pos_weight_tensor.shape)
print("pos_weight tensor device:", pos_weight_tensor.device)

# Save weight table for documentation
weight_table.to_csv(OUTPUT_DIR / "sqrt_pos_weight_table.csv", index=False)
print("\nSaved:")
print(OUTPUT_DIR / "sqrt_pos_weight_table.csv")

Training patch distribution:
Total train patches: 1359460
Normal train patches: 1334601
Abnormal train patches: 24859
Abnormal percentage: 1.8286 %

Dynamic 1:1 sampler:
num_train_samples per epoch: 49718
Expected abnormal samples per epoch: 24859
Expected normal samples per epoch: 24859

Square-root pos_weight table:


,OED_feature,positive_count_train,raw_pos_weight,sqrt_pos_weight
0,Irregular epithelial stratification,435,113.294253,10.643977
1,Loss of polarity of basal cells,1975,24.173671,4.916673
2,Drop shaped rete ridges,717,68.341701,8.266904
3,Premature keratinization in single cells,7430,5.691521,2.385691
4,Loss of epithelial cell cohesion,5460,8.105861,2.847079
5,Abnormal variation in nuclear size,5337,8.315720,2.883699
6,Abnormal variation in nuclear shape,7879,5.310192,2.304385
7,Abnormal variation in cell size,317,155.839116,12.483554
8,Abnormal variation in cell shape,8548,4.816331,2.194614
9,Increased N:C ratio,3236,14.364030,3.789991



pos_weight tensor shape: torch.Size([12])
pos_weight tensor device: cuda:0

Saved:
/home/user/jiangjie/Jiangjie_Project/outputs/kee_reproduction/controlled_hard_seed42_50_sqr_no_aug_f1select/sqrt_pos_weight_table.csv


In [8]:
# =========================
# Create Dataset and DataLoader + real batch check
# =========================

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

# Current augmentation setting: only one item, "None"
aug_name, train_transform = list(AUGMENTATION_COMBINATIONS.items())[0]

print("Current augmentation:", aug_name)
assert aug_name == "None", "Run A should use no augmentation."

train_dataset = MultiLabelTileDataset(
    train_df,
    label_columns=label_columns,
    transform=train_transform
)

val_dataset = MultiLabelTileDataset(
    val_df,
    label_columns=label_columns,
    transform=val_test_transform
)

test_dataset = MultiLabelTileDataset(
    test_df,
    label_columns=label_columns,
    transform=val_test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=loader_generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=loader_generator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=loader_generator
)

print("Dataset sizes:")
print("train_dataset:", len(train_dataset))
print("val_dataset:", len(val_dataset))
print("test_dataset:", len(test_dataset))

print("\nDataLoader batches:")
print("train_loader batches per epoch:", len(train_loader))
print("val_loader batches:", len(val_loader))
print("test_loader batches:", len(test_loader))

# Check one real training batch
images, labels = next(iter(train_loader))

print("\nOne real training batch:")
print("images shape:", images.shape)
print("labels shape:", labels.shape)
print("images dtype:", images.dtype)
print("labels dtype:", labels.dtype)
print("labels min/max:", labels.min().item(), labels.max().item())
print("positive labels in this batch:", int(labels.sum().item()))

assert images.shape[1:] == (3, 224, 224)
assert labels.shape[1] == NUM_CLASSES
assert labels.min().item() >= 0
assert labels.max().item() <= 1

print("\nDataLoader batch check passed.")

Current augmentation: None
Dataset sizes:
train_dataset: 1359460
val_dataset: 198083
test_dataset: 314284

DataLoader batches:
train_loader batches per epoch: 389
val_loader batches: 1548
test_loader batches: 2456

One real training batch:
images shape: torch.Size([128, 3, 224, 224])
labels shape: torch.Size([128, 12])
images dtype: torch.float32
labels dtype: torch.float32
labels min/max: 0.0 1.0
positive labels in this batch: 103

DataLoader batch check passed.


In [9]:
# @title Hard-label F1-selection training and evaluation
# --- 4. Main Training and Evaluation Loop ---
# This version keeps the original hard-label baseline data pipeline,
# but changes checkpoint selection from Val Macro AUC to Val Hard Macro F1@0.5.

def train_and_evaluate_run(model_name, train_loader, val_loader, test_loader, num_classes, criterion,
                           optimizer_class, lr, num_epochs, patience, label_columns,
                           checkpoint_path, weight_type, covertype):

    print(f"\n--- Running Augmentation Strategy: {model_name} ---")
    print("Training loss: original 50% hard labels")
    print("Validation loss: original 50% hard labels")
    print("Validation/test metrics: original 50% hard labels")
    print("Primary checkpoint selection: Val Hard Macro F1@0.5")

    # Initialize Model for a fresh run, loading checkpoint inside ctranspath
    model = ctranspath(num_classes, checkpoint_path=checkpoint_path).to(device)
    optimizer = optimizer_class(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    threshold = 0.5
    best_val_macro_f1 = -1.0
    epochs_without_improvement = 0

    best_model_path = str(
        OUTPUT_DIR / f"best_model_{weight_type}_{covertype}_{model_name}_by_macro_f1.pth"
    )

    history_rows = []

    for epoch in range(num_epochs):
        # -------------------------
        # Training phase
        # -------------------------
        model.train()
        running_loss = 0.0

        for inputs, labels in tqdm(train_loader, desc=f"E {epoch+1}/{num_epochs} (Train Hard)"):
            if inputs is None:
                continue

            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        # Keep the original baseline denominator for consistency with the source notebook.
        train_loss = running_loss / len(train_loader.dataset)

        # -------------------------
        # Validation phase
        # -------------------------
        model.eval()
        val_true = []
        val_probs = []
        val_loss = 0.0

        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f"E {epoch+1}/{num_epochs} (Val Hard)"):
                if inputs is None:
                    continue

                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)

                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)

                probs = torch.sigmoid(outputs)
                val_true.extend(labels.cpu().numpy())
                val_probs.extend(probs.cpu().numpy())

        val_loss /= len(val_loader.dataset)

        val_true = np.array(val_true).astype(int)
        val_probs = np.array(val_probs)
        val_pred_binary = (val_probs >= threshold).astype(int)

        val_micro_auc = roc_auc_score(val_true, val_probs, average="micro")
        val_macro_auc = roc_auc_score(val_true, val_probs, average="macro")

        val_micro_f1 = f1_score(val_true, val_pred_binary, average="micro", zero_division=0)
        val_macro_f1 = f1_score(val_true, val_pred_binary, average="macro", zero_division=0)

        print(
            f"| Epoch {epoch+1:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Hard Micro AUC: {val_micro_auc:.4f} | "
            f"Val Hard Macro AUC: {val_macro_auc:.4f} | "
            f"Val Hard Micro F1@0.5: {val_micro_f1:.4f} | "
            f"Val Hard Macro F1@0.5: {val_macro_f1:.4f} |"
        )

        history_rows.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_micro_auc": val_micro_auc,
            "val_macro_auc": val_macro_auc,
            "val_micro_f1_at_0_5": val_micro_f1,
            "val_macro_f1_at_0_5": val_macro_f1,
        })

        # -------------------------
        # Early stopping based on Val Hard Macro F1@0.5
        # -------------------------
        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_macro_f1
            torch.save(model.state_dict(), best_model_path)
            epochs_without_improvement = 0
            print(f"Saved best Macro F1 model to: {best_model_path}")
        else:
            epochs_without_improvement += 1
            print(f"No Macro F1 improvement. epochs_without_improvement = {epochs_without_improvement}")

            if epochs_without_improvement >= patience:
                print(f"Early stopping triggered. Best Macro F1 model saved at: {best_model_path}\n")
                break

    # Save training history
    history_df = pd.DataFrame(history_rows)
    history_path = OUTPUT_DIR / f"training_history_{weight_type}_{covertype}_{model_name}_by_macro_f1.csv"
    history_df.to_csv(history_path, index=False)
    print("Saved training history:", history_path)

    # -------------------------
    # Final evaluation on hard test set
    # -------------------------
    print("Loading best Macro F1 checkpoint for final test evaluation:", best_model_path)
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    model.eval()

    test_true = []
    test_probs = []

    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc=f"Testing {model_name}"):
            if inputs is None:
                continue

            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            probs = torch.sigmoid(outputs)

            test_true.extend(labels.cpu().numpy())
            test_probs.extend(probs.cpu().numpy())

    test_true = np.array(test_true).astype(int)
    test_probs = np.array(test_probs)
    test_pred_binary = (test_probs >= threshold).astype(int)

    # AUROC
    test_micro_auc = roc_auc_score(test_true, test_probs, average="micro")
    test_macro_auc = roc_auc_score(test_true, test_probs, average="macro")

    # Overall hard-label metrics
    micro_precision = precision_score(test_true, test_pred_binary, average="micro", zero_division=0)
    micro_recall = recall_score(test_true, test_pred_binary, average="micro", zero_division=0)
    micro_f1 = f1_score(test_true, test_pred_binary, average="micro", zero_division=0)

    macro_precision = precision_score(test_true, test_pred_binary, average="macro", zero_division=0)
    macro_recall = recall_score(test_true, test_pred_binary, average="macro", zero_division=0)
    macro_f1 = f1_score(test_true, test_pred_binary, average="macro", zero_division=0)

    weighted_precision = precision_score(test_true, test_pred_binary, average="weighted", zero_division=0)
    weighted_recall = recall_score(test_true, test_pred_binary, average="weighted", zero_division=0)
    weighted_f1 = f1_score(test_true, test_pred_binary, average="weighted", zero_division=0)

    # Per-label metrics
    per_class_rows = []

    print("\n=== Per-label AUROC and Binary Metrics (Hard Test Set, threshold=0.5) ===")
    for i, label in enumerate(label_columns):
        y_true = test_true[:, i]
        y_prob = test_probs[:, i]
        y_pred = test_pred_binary[:, i]

        try:
            auc = roc_auc_score(y_true, y_prob)
        except ValueError:
            auc = np.nan

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        row = {
            "Feature": label,
            "AUROC": auc,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "Support": int(y_true.sum()),
            "Predicted Positive": int(y_pred.sum()),
        }
        per_class_rows.append(row)

    per_class_df = pd.DataFrame(per_class_rows)
    display(per_class_df)

    print(f"\n--- Overall Classification Report for {model_name} (Hard Test Set) ---")
    print(
        classification_report(
            y_true=test_true,
            y_pred=test_pred_binary,
            target_names=label_columns,
            zero_division=0,
            output_dict=False,
        )
    )

    summary_df = pd.DataFrame([{
        "Strategy": model_name,
        "threshold": threshold,
        "micro_auroc": test_micro_auc,
        "macro_auroc": test_macro_auc,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,
        "weighted_f1": weighted_f1,
    }])

    print(f"\n--- Final Test Results for {model_name} ---")
    display(summary_df)
    print("-------------------------------------------------\n")

    # Save outputs
    summary_path = OUTPUT_DIR / f"test_summary_{weight_type}_{covertype}_{model_name}_by_macro_f1.csv"
    per_class_path = OUTPUT_DIR / f"test_per_class_{weight_type}_{covertype}_{model_name}_by_macro_f1.csv"
    probs_path = OUTPUT_DIR / f"test_probs_{weight_type}_{covertype}_{model_name}_by_macro_f1.npy"
    targets_path = OUTPUT_DIR / f"test_targets_hard_{weight_type}_{covertype}_{model_name}_by_macro_f1.npy"

    summary_df.to_csv(summary_path, index=False)
    per_class_df.to_csv(per_class_path, index=False)
    np.save(probs_path, test_probs)
    np.save(targets_path, test_true)

    print("Saved test summary:", summary_path)
    print("Saved per-class metrics:", per_class_path)
    print("Saved test probabilities:", probs_path)
    print("Saved hard test targets:", targets_path)

    return summary_df.iloc[0].to_dict()


# Result of AUC 
Test Micro AUROC
Test Macro AUROC

In [10]:
set_seed(SEED)

In [11]:
# @title 50% - Square root Weight
# --- 5. Execution Block ---
base_path = str(RP50_DIR)
weight_type = "Sqr"
covertype = "50"

if __name__ == '__main__':
    # Load your DataFrames
    try:
        train_files = [f"{base_path}/final_df_train{i}.csv" for i in range(1, 11)]
        # train_df = pd.concat(
        #     [pd.read_csv(f, low_memory=False) for f in train_files],
        #     ignore_index=True
        # )
        # val_df = pd.read_csv(base_path +'/final_df_val.csv', low_memory=False)
        # test_df = pd.read_csv(base_path +'/final_df_test.csv', low_memory=False)
        # train_df = pd.read_csv(base_path +'/demo_df_train.csv')
        # val_df = pd.read_csv(base_path +'/demo_df_val.csv', low_memory=False)
        # test_df = pd.read_csv(base_path +'/demo_df_test.csv', low_memory=False)
        train_df = pd.concat(
            [pd.read_csv(f, low_memory=False) for f in train_files],
            ignore_index=True
        )
        val_df = pd.read_csv(base_path +'/final_df_val.csv', low_memory=False)
        test_df = pd.read_csv(base_path +'/final_df_test.csv', low_memory=False)

        # Replace Colab-style filepath with DICC filepath
        old_prefix = "/content/ResearchProject_50"
        new_prefix = str(RP50_DIR)

        for df in [train_df, val_df, test_df]:
            df["filepath"] = df["filepath"].astype(str).str.replace(
                old_prefix,
                new_prefix,
                regex=False
            )

        print("\nExample filepath after replacement:")
        print(train_df["filepath"].iloc[0])
        print("Exists:", Path(train_df["filepath"].iloc[0]).exists())

        assert Path(train_df["filepath"].iloc[0]).exists(), "Filepath replacement failed."
    except FileNotFoundError:
        print("ERROR: Please ensure df_train.csv, df_val.csv, and df_test.csv are in the current directory.")
        exit()

    # --- 2. Calculate pos_weight for BCEWithLogitsLoss (from IMbalanced data) ---
    print("Calculating pos_weight for loss function (from full imbalanced set)...")

    # Sum of positive labels for each class
    # print("Training")
    positive_counts = train_df[label_columns].sum()
    print(positive_counts)
    # print("Validation")
    # positive_counts1 = val_df[label_columns].sum()
    # print(positive_counts1)
    # print("Testing")
    # positive_counts2 = test_df[label_columns].sum()
    # print(positive_counts2)

    # Find the total number of 'normal' (all-zero) patches
    print("\n--- Setting up Dynamic 1:1 Sampler ---")

    

    # Identify abnormal (at least one label) vs normal (no labels)
    is_abnormal = train_df[label_columns].sum(axis=1) > 0
    n_abnormal = is_abnormal.sum()
    n_normal = len(train_df) - n_abnormal

    print(f"Dataset Counts: {n_abnormal} Abnormal, {n_normal} Normal")

    # Create weights for the SAMPLER (to force 1:1 balance)
    sample_weights = torch.zeros(len(train_df))
    sample_weights[is_abnormal] = 1.0 / n_abnormal
    sample_weights[~is_abnormal] = 1.0 / n_normal

    # Define the length of one epoch (2 * n_abnormal ensures we see all abnormal patches)
    num_train_samples = int(n_abnormal * 2)

    # Initialize the Sampler
    sampler_generator = torch.Generator()
    sampler_generator.manual_seed(SEED)

    sampler = WeightedRandomSampler(
        weights=sample_weights.double(),
        num_samples=num_train_samples,
        replacement=True,
        generator=sampler_generator
    )
    print(f"Sampler initialized. Epoch length: {num_train_samples} patches.")

    def seed_worker(worker_id):
        worker_seed = SEED + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    loader_generator = torch.Generator()
    loader_generator.manual_seed(SEED)

    # ==============================================================================
    # 2. CALCULATE DAMPENED POS_WEIGHT (PASTE THE NEW CODE HERE)
    # ==============================================================================
    print("\n--- Calculating Dampened pos_weight for Loss Function ---")

    # 1. Get the intrinsic count of positive labels
    positive_counts = train_df[label_columns].sum()

    # 2. Calculate "Effective Negatives" based on the Sampler's 1:1 output
    #    In one epoch, total samples = (2 * n_abnormal)
    effective_negatives = (2 * n_abnormal) - positive_counts

    # 3. Calculate RAW weights first
    raw_weights = effective_negatives / (positive_counts + 1e-6)

    print("Raw Adjusted Weights (Too Aggressive):")
    print(raw_weights)

    # 4. THE FIX: Dampen the weights using Square Root
    #    This prevents the model from predicting "1" for everything.
    pos_weight = torch.sqrt(torch.tensor(raw_weights.values, dtype=torch.float32))

    # 5. Create the Loss Function
    pos_weight = pos_weight.to(device)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    print("\nDampened Weights (Using Sqrt):")
    for label, w in zip(label_columns, pos_weight):
        print(f"  '{label}': {w:.2f}")

    # Initialize Test and Validation Loaders (fixed transform)
    val_dataset = MultiLabelTileDataset(val_df, label_columns, transform=val_test_transform)
    test_dataset = MultiLabelTileDataset(test_df, label_columns, transform=val_test_transform)
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn,
        worker_init_fn=seed_worker,
        generator=loader_generator
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn,
        worker_init_fn=seed_worker,
        generator=loader_generator
    )

    final_results = {}

    # --- Run Loop for Hyperparameter Tuning ---
    for name, train_transform in AUGMENTATION_COMBINATIONS.items():
        # Setup TRAIN Dataset and Loader for current augmentation
        train_dataset = MultiLabelTileDataset(train_df, label_columns, transform=train_transform)
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=NUM_WORKERS,
            collate_fn=collate_fn,
            shuffle=False,
            worker_init_fn=seed_worker,
            generator=loader_generator
        )

        test_summary = train_and_evaluate_run(
            model_name=name,
            train_loader=train_loader,
            val_loader=val_loader,
            test_loader=test_loader,
            num_classes=NUM_CLASSES,
            criterion=criterion,
            optimizer_class=torch.optim.AdamW,
            lr=LEARNING_RATE,
            num_epochs=EPOCHS,
            patience=PATIENCE,
            label_columns=label_columns,
            checkpoint_path=CHECKPOINT_PATH, # Passes the manual checkpoint path
            weight_type=weight_type,
            covertype=covertype
        )

        final_results[name] = test_summary

    # --- Print and save final summary ---
    final_results_df = pd.DataFrame(list(final_results.values()))

    print("\n\n================== FINAL TEST SUMMARY ==================")
    display(final_results_df)

    final_results_path = OUTPUT_DIR / "final_results_summary.csv"
    final_results_df.to_csv(final_results_path, index=False)

    print("Saved final results:", final_results_path)
    print("=======================================================")


Example filepath after replacement:
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/266-DP-14/266-DP-14_tile_0000000.jpg
Exists: True
Calculating pos_weight for loss function (from full imbalanced set)...
Irregular epithelial stratification          435
Loss of polarity of basal cells             1975
Drop shaped rete ridges                      717
Premature keratinization in single cells    7430
Loss of epithelial cell cohesion            5460
Abnormal variation in nuclear size          5337
Abnormal variation in nuclear shape         7879
Abnormal variation in cell size              317
Abnormal variation in cell shape            8548
Increased N:C ratio                         3236
Increased number and size of nucleoli       3963
Hyperchromasia                              6344
dtype: int64

--- Setting up Dynamic 1:1 Sampler ---
Dataset Counts: 24859 Abnormal, 1334601 Normal
Sampler initialized. Epoch length: 49718 patches.

--- Calculating Dampened pos_weight for Lo

E 1/50 (Val Hard): 100%|██████████| 1548/1548 [05:27<00:00,  4.73it/s]


| Epoch 01 | Train Loss: 0.0167 | Val Loss: 0.2170 | Val Hard Micro AUC: 0.8600 | Val Hard Macro AUC: 0.8529 | Val Hard Micro F1@0.5: 0.0537 | Val Hard Macro F1@0.5: 0.0572 |
Saved best Macro F1 model to: /home/user/jiangjie/Jiangjie_Project/outputs/kee_reproduction/controlled_hard_seed42_50_sqr_no_aug_f1select/best_model_Sqr_50_None_by_macro_f1.pth


E 2/50 (Val Hard): 100%|██████████| 1548/1548 [05:28<00:00,  4.71it/s]


| Epoch 02 | Train Loss: 0.0147 | Val Loss: 0.2163 | Val Hard Micro AUC: 0.8457 | Val Hard Macro AUC: 0.8353 | Val Hard Micro F1@0.5: 0.0424 | Val Hard Macro F1@0.5: 0.0545 |
No Macro F1 improvement. epochs_without_improvement = 1


...


--- Overall Classification Report for None (Hard Test Set) ---
                                          precision    recall  f1-score   support

     Irregular epithelial stratification       0.00      0.00      0.00        53
         Loss of polarity of basal cells       0.00      0.00      0.00       212
                 Drop shaped rete ridges       0.00      0.00      0.00        45
Premature keratinization in single cells       0.22      0.22      0.22      3467
        Loss of epithelial cell cohesion       0.01      0.02      0.02      1629
      Abnormal variation in nuclear size       0.00      0.01      0.00       145
     Abnormal variation in nuclear shape       0.02      0.19      0.04      1424
         Abnormal variation in cell size       0.00      0.00      0.00        20
        Abnormal variation in cell shape       0.03      0.20      0.05      1992
                     Increased N:C ratio       0.00      0.01      0.00       380
   Increased number and size of n

,Strategy,threshold,micro_auroc,macro_auroc,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1
0,None,0.5,0.898422,0.86126,0.025233,0.214328,0.04515,0.029573,0.109394,0.037209,0.083924,0.214328,0.102092


-------------------------------------------------

Saved test summary: /home/user/jiangjie/Jiangjie_Project/outputs/kee_reproduction/controlled_hard_seed42_50_sqr_no_aug_f1select/test_summary_Sqr_50_None_by_macro_f1.csv
Saved per-class metrics: /home/user/jiangjie/Jiangjie_Project/outputs/kee_reproduction/controlled_hard_seed42_50_sqr_no_aug_f1select/test_per_class_Sqr_50_None_by_macro_f1.csv
Saved test probabilities: /home/user/jiangjie/Jiangjie_Project/outputs/kee_reproduction/controlled_hard_seed42_50_sqr_no_aug_f1select/test_probs_Sqr_50_None_by_macro_f1.npy
Saved hard test targets: /home/user/jiangjie/Jiangjie_Project/outputs/kee_reproduction/controlled_hard_seed42_50_sqr_no_aug_f1select/test_targets_hard_Sqr_50_None_by_macro_f1.npy


================== FINAL TEST SUMMARY ==================


,Strategy,threshold,micro_auroc,macro_auroc,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1
0,None,0.5,0.898422,0.86126,0.025233,0.214328,0.04515,0.029573,0.109394,0.037209,0.083924,0.214328,0.102092


Saved final results: /home/user/jiangjie/Jiangjie_Project/outputs/kee_reproduction/controlled_hard_seed42_50_sqr_no_aug_f1select/final_results_summary.csv


In [12]:
print("--- Positive Class Counts in VALIDATION Set ---")
print(val_df[label_columns].sum())

print("--- Positive Class Counts in Testing Set ---")
print(test_df[label_columns].sum())

--- Positive Class Counts in VALIDATION Set ---
Irregular epithelial stratification          376
Loss of polarity of basal cells              357
Drop shaped rete ridges                       86
Premature keratinization in single cells     449
Loss of epithelial cell cohesion             197
Abnormal variation in nuclear size          2313
Abnormal variation in nuclear shape         1924
Abnormal variation in cell size             1316
Abnormal variation in cell shape            1461
Increased N:C ratio                         1554
Increased number and size of nucleoli       1230
Hyperchromasia                               768
dtype: int64
--- Positive Class Counts in Testing Set ---
Irregular epithelial stratification           53
Loss of polarity of basal cells              212
Drop shaped rete ridges                       45
Premature keratinization in single cells    3467
Loss of epithelial cell cohesion            1629
Abnormal variation in nuclear size           145
Abnormal var